# 📄 PDF Clinical Report Generator

> Tool C — 為單一 cell line 生成 2 頁 A4 PDF 臨床報告

輸入 `cell_id` → **2 秒內** 生成可印的 PDF。適合帶入分子腫瘤委員會(MTB)討論。

PDF 內容:
- Page 1:Cell line 資訊 + Top 15 推薦藥物表(P, CI, target, pathway)
- Page 2:結果判讀指引 + 模型限制揭露
- 全頁:**「Research Use Only」紅字免責聲明**

## Cell 1 — 環境準備(同 Widget notebook)

In [ ]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules
print(f'Environment: {"Colab" if IN_COLAB else "Local"}')

if IN_COLAB:
    GITHUB_USER = 'otonifrio2812'  # (已預填您的帳號)
    REPO_NAME = 'pediatric-bt-drug-prediction'

    if not os.path.exists(REPO_NAME):
        !git clone -q https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_NAME}
    !pip install -q -r requirements.txt

    if not os.path.exists('intermediates/stage6_ensemble_models.pkl'):
        print('Downloading intermediates from GitHub Release...')
        os.makedirs('intermediates', exist_ok=True)
        RELEASE_URL = (
            f'https://github.com/{GITHUB_USER}/{REPO_NAME}/'
            'releases/download/v1.0.0/intermediates.zip'
        )
        !wget -q {RELEASE_URL} -O intermediates.zip
        !unzip -q intermediates.zip -d intermediates/
        print('✓ Intermediates ready')
else:
    if not os.path.exists('src'):
        os.chdir('..')
    print(f'Working directory: {os.getcwd()}')

sys.path.insert(0, '.')

## Cell 2 — 載入模型

In [ ]:
from src.drug_ranking import load_artifacts
from src.pdf_report import generate_clinical_report

print('Loading artifacts (預計 30 秒)...')
artifacts = load_artifacts(intermediates_dir='intermediates/')
print(f'✓ {len(artifacts.cell_metadata_lookup)} cells × '
      f'{len(artifacts.drug_features)} drugs loaded')

## Cell 3 — 生成單個報告(快速 demo)

用 SF539(SIDM00083,典型 GBM cell line)做示範。

In [ ]:
# 生成 SF539 報告
import os
os.makedirs('results/reports', exist_ok=True)

report_path = generate_clinical_report(
    cell_id='SIDM00083',
    artifacts=artifacts,
    top_k=15,
    output_path='results/reports/SF539_GBM_report.pdf',
)
print(f'✓ Report generated: {report_path}')
print(f'  File size: {report_path.stat().st_size / 1024:.1f} KB')

# 在 Colab 下載產出的 PDF
if IN_COLAB:
    from google.colab import files
    files.download(str(report_path))

## Cell 4 — 批次生成多個報告

對 3 種癌種各挑 1 個代表 cell,各生成一份報告。

In [ ]:
from src.drug_ranking import list_cells_by_cancer_type

cells_by_type = list_cells_by_cancer_type(artifacts)

# 每種癌種挑第一個 cell 作 demo
demo_cells = {}
for ctype in ['Glioblastoma', 'Glioma', 'Neuroblastoma']:
    if ctype in cells_by_type:
        cid, cname = cells_by_type[ctype][0]
        demo_cells[ctype] = (cid, cname)

print('Generating reports for demo cells...\n')
for ctype, (cid, cname) in demo_cells.items():
    print(f'  {ctype}: {cid} ({cname})')
    safe_name = cname.replace('/', '_').replace(' ', '_')
    path = generate_clinical_report(
        cell_id=cid,
        artifacts=artifacts,
        top_k=15,
        output_path=f'results/reports/{ctype}_{safe_name}_report.pdf',
    )
    print(f'    💾 {path.name} ({path.stat().st_size / 1024:.1f} KB)')

print(f'\n✓ {len(demo_cells)} reports generated in results/reports/')

## Cell 5 — (選用)自訂報告

您可以對任何 cell 生成報告,只要它的 `SANGER_MODEL_ID` 在資料集裡:

In [ ]:
# 列出資料集中所有可用的 cell
available_cells = sorted(artifacts.cell_metadata_lookup.keys())
print(f'共 {len(available_cells)} 個 cells 可用,前 10 個:')
for cid in available_cells[:10]:
    m = artifacts.cell_metadata_lookup[cid]
    print(f'  {cid:10s}  {m["CELL_LINE_NAME"]:15s}  {m["CANCER_TYPE"]}')

print('\n要為自己選的 cell 生成報告,把下面 cell_id 改掉執行即可:')
print('''
    path = generate_clinical_report(
        cell_id='SIDM00XXX',     # ← 改這裡
        artifacts=artifacts,
        top_k=20,                 # 改 Top K
        output_path='my_report.pdf',
    )
''')